# Autoresearch

An agent edits one file, measures, keeps or discards — and repeats without you.

*You are the slow part of research, not the thinking.*

> Faithful to [karpathy/autoresearch](https://github.com/karpathy/autoresearch): one loop.

`solve(n)` counts the primes below `n`, from the sieve everyone writes first. No closed form to
jump to, so the loop has to climb.

## 0. Setup

Reading and writing are separate permissions — this split *is* the design:

| path | reads | writes |
|---|---|---|
| `program.md` | **yes** — every prompt | no |
| `src/` | **yes** | **yes** — on a win |
| `harness/` | no | no |

In [1]:
import os
import re
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from harness.measure import measure   # protected/ — never enters a prompt

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"

SOLVE = Path("src/solve.py")   # the only file the agent may rewrite
PROGRAM = Path("program.md").read_text()
MAX_EXPERIMENTS, PATIENCE = 15, 8
MIN_GAIN = 0.95   # a win must beat best by 5% — under that, the timer can't tell

## 1. The boundary — load both and look

The agent reads `program.md` and `src/`. It never sees `harness/`, so the verifier can't be
argued with.

In [2]:
for label, d in [("EDITABLE  src/  — the agent rewrites these", "src"),
                 ("PROTECTED harness/  — never enters a prompt", "harness")]:
    print(f"{'='*70}\n{label}\n{'='*70}")
    for f in sorted(p for p in Path(d).iterdir() if p.is_file()):
        print(f"--- {f} ---\n{f.read_text().rstrip()}\n")

EDITABLE  src/  — the agent rewrites these
--- src/solve.py ---
# EDITABLE — autoresearch rewrites this in place. Karpathy's train.py.


def solve(n):
    sieve = [True] * n
    sieve[0] = sieve[1] = False
    i = 2
    while i * i < n:
        if sieve[i]:
            for j in range(i * i, n, i):
                sieve[j] = False
        i += 1
    return sum(sieve)

PROTECTED harness/  — never enters a prompt
--- harness/cases.json ---
{
  "_note": "count of primes below n — n is exclusive. pi(7)=3 counts 2,3,5 and NOT 7. This is the spec the model keeps getting wrong, and only a prime n catches it.",
  "tests": [[7, 3], [10, 4], [100, 25], [1000, 168], [10000, 1229]],
  "workload": 2000000,
  "expected": 148933
}

--- harness/measure.py ---
"""PROTECTED — never enters a prompt. Karpathy's `prepare.py`.

An agent that can edit the verifier optimises the verifier: it deletes the failing
case instead of passing it.
"""

import json
import time
from pathlib import Path

_CASES = json.loa

## 2. The prompt

Everything the agent gets: `program.md`, the editable file, the last measurement. Nothing else.

In [3]:
def propose(src, seconds, note):
    """The only thing the agent ever sees: program.md + solve.py + the last measurement."""
    msg = (f"{PROGRAM}\n\n"
           f"src/solve.py:\n```python\n{src}\n```\n"
           f"Current: {seconds*1000:.3f} ms. Last result: {note}\n"
           f"Propose ONE change. Reply with only the new `def solve(n):` in a ```python block.")
    out = client.chat.completions.create(
        model=MODEL, max_tokens=700, messages=[{"role": "user", "content": msg}],
    ).choices[0].message.content
    m = re.search(r"```python\n(.*?)```", out, re.S)
    return m.group(1).strip() if m else None

## 3. The loop

Measure, then write only on a win — so `src/solve.py` only ever holds the best. A win must
clear 5%: `measure` already takes the min of 3 runs, and anything smaller is still the timer.

In [4]:
def autoresearch():
    best = SOLVE.read_text()
    base_s, note = measure(best)       # measured once — every x below is against this
    best_s, stale = base_s, 0
    print(f"baseline {base_s*1000:.1f} ms\n")

    for i in range(1, MAX_EXPERIMENTS + 1):
        cand = propose(best, best_s, note)
        secs, note = measure(cand) if cand else (None, "no code block")
        kept = secs is not None and secs < best_s * MIN_GAIN   # 5% — not noise

        if kept:
            SOLVE.write_text(cand)          # commit: the file only ever holds the best
            best, best_s, stale = cand, secs, 0
        else:
            stale += 1                      # discard: nothing was written, nothing to undo

        # a time means it passed; no time means the note says why not
        print(f"  {i:>2}  {('%.1f ms' % (secs * 1000)) if secs else note[:44]:<22}"
              f"{'KEEP  %.1fx' % (base_s / secs) if kept else ''}")

        if stale >= PATIENCE:
            print(f"  stop: no improvement in {PATIENCE} experiments")
            break
    return best, best_s, base_s

## 4. Run

`git diff src/` afterwards to see what it did.

In [5]:
best, best_s, base_s = autoresearch()

print(f"\nbaseline {base_s*1000:.1f} ms -> best {best_s*1000:.1f} ms  ({base_s/best_s:.1f}x)")
print(best)

baseline 67.0 ms



   1  67.0 ms               


   2  71.9 ms               


   3  71.6 ms               


   4  26.6 ms               KEEP  2.5x


...

   8  27.1 ms               


   9  WRONG on n=7: got 4, want 3


  10  32.5 ms               


  11  27.4 ms               


  12  30.4 ms               
  stop: no improvement in 8 experiments

baseline 67.0 ms -> best 26.6 ms  (2.5x)
def solve(n):
    if n < 3:
        return 0
    sieve = [True] * (n // 2)
    sieve[0] = False
    for i in range(1, int(n ** 0.5) // 2 + 1):
        if sieve[i]:
            for j in range(2 * i * (i + 1), len(sieve), 2 * i + 1):
                sieve[j] = False
    return sum(sieve) + 1


## 5. The ceiling

A speedup means nothing without one. Scored by the same `measure`; never enters a prompt.

In [6]:
CEILING = """
def solve(n):
    sieve = bytearray([1]) * (n // 2)        # a bytearray, not a list of bools
    sieve[0] = 0
    i = 3
    while i * i < n:
        if sieve[i // 2]:
            start = i * i // 2
            sieve[start::i] = bytearray(len(range(start, n // 2, i)))
        i += 2
    return sum(sieve) + 1
"""

s, note = measure(CEILING)         # same gate: correct before it is timed
print(f"the loop, 15 experiments   {best_s*1000:>5.1f} ms  {base_s/best_s:>5.1f}x")
print(f"hand-written ceiling       {s*1000:>5.1f} ms  {base_s/s:>5.1f}x  {note}")

the loop, 15 experiments    26.6 ms    2.5x
hand-written ceiling         3.3 ms   20.2x  ok


## Key findings

**One win in fifteen** — experiment 4, `2.5x`, and `PATIENCE` stopped it at 12. §5 scores
`20.2x` through the same gate, so the loop took about an eighth of what was there.

**`n=7` rejected three candidates** (5, 7, 9), every one the same `<= n` off-by-one. The other
cases are all composite and cannot see it — that single case is the whole gate.

**The trace is a sample, not a result.** This same code and model reached `9.0x` in four wins an
hour earlier, and `2.5x` in one here. Proposals are sampled; what reproduces is the mechanism,
not the number.

## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **One run is an anecdote** | Proposals are sampled, so the trace never repeats: `2.5x` in one win here, `9.0x` in four an hour earlier, same code and model | Run it N times and report the spread — a single trace shows the mechanism, not the number |
| **The model is the ceiling** | It reaches odds-only, and sometimes slice assignment, but never `bytearray` — §5's `20.2x` stays out of reach either way | Use a model that has the idea |
| **No memory** | Each proposal sees only the best and the last note, so it repeats itself — the same `n=7` off-by-one at 5, 7 and 9 | Pass the log |
| **The gate is hand-tuned** | Min-of-3 still leaves a few percent, and `MIN_GAIN=0.95` sits just above it — both picked for this box. Elsewhere a real win could fall under it | Re-time one candidate N times; set the gate above the spread |
| **The boundary is convention** | `exec` runs model code in-process with write access to `harness/` | Subprocess, read-only mount, timeout |
| **The metric is the objective** | Correctness is pass/fail, no partial credit. `n=7` is a case only because every other one is composite and can't see a `<= n` off-by-one — it rejected three candidates | Verify the verifier first |
| **No rollback** | The first win overwrites the baseline in place | Commit before running — this ate the scaffold twice |
| **One loop, by design** | Nothing notices that experiments 1–3 were going nowhere | none — a second loop is a different mechanism |